In [ ]:
import os, glob, time, json, re, random, shutil
import base64, cv2
import pandas as pd

from typing import Dict, Any
from copy import deepcopy
from datetime import datetime

import torch
import torch.nn.functional as F

from openai import OpenAI
import anthropic
from pydantic import BaseModel, conlist
from typing import List, Literal, Optional

import utils

API_KEY_JSON_PATH = "APIKEY/api_key.json"
API_KEY_JSON = json.load(open(API_KEY_JSON_PATH, "r"))
OPENAI_API_KEY = API_KEY_JSON["OpenAI_yong"]
CLAUDE_API_KEY = API_KEY_JSON["Anthropic_yong"]

In [ ]:
agent = OpenAI(api_key=OPENAI_API_KEY)
model_name = "gpt-5.4"

In [ ]:
df_class_path = "data/action_classes.csv"
df_class = pd.read_csv(df_class_path)
action_classes = df_class["class"].tolist()
action_classes.append("none")

prompt = f"""
# ROLE:
You are an expert construction manager analyzing video frames from construction site videos.

# INSTRUCTION:
Determine the worker's current action from the provided frame or frames.

An action should be identified ONLY when the worker is actively performing a construction task through direct physical interaction with relevant work entities, such as:
- tools
- materials
- construction equipment
- building components

Direct physical interaction means the worker's hand(s) are visibly in contact with the object(s) involved in the task.

Return "none" in the following cases:
- the worker is preparing to start a task
- the worker is observing, waiting, or idle
- the worker's hands are not in active contact with relevant work objects
- the visible action does not clearly match any provided action options
- the worker is performing a non-target action outside the provided action list

Choose ONLY from the provided action options below:
{action_classes}

Be conservative. If the action is ambiguous or insufficiently visible, return "none".

# OUTPUT FORMAT:
Return a valid JSON object only.
Key: "action"
Value: One action from the provided choices or "none"

# EXAMPLE:
Input: A frame showing a worker using a hammer to drive a nail into a piece of wood
Output: "action": "hammer"
"""

class StructuredResponse(BaseModel):
    action: conlist(Literal[tuple(action_classes)], min_length=1, max_length=1)

In [ ]:
frame_path = "data/frames_fps1/clipped_0_11_cart_v1_8_214.jpg"
base64_frame = utils.encode_image(frame_path)

# response = agent.responses.create(
#     model=model_name,
#     input=[
#         {
#             "role": "user",
#             "content": [
#                 { "type": "input_text", "text": prompt },
#                 {
#                     "type": "input_image",
#                     "image_url": f"data:image/jpeg;base64,{base64_frame}",
#                     "detail": "auto"
#                 },
#             ],
#         }
#     ],
# )
response = agent.responses.parse(
    model=model_name,
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": prompt },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_frame}",
                    "detail": "auto"
                },
            ],
        }
    ],
    text_format=StructuredResponse,
)

In [ ]:
agent = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

In [ ]:
frame_path = "data/frames_fps1/clipped_0_11_cart_v1_8_214.jpg"
base64_frame = utils.encode_image(frame_path)
image_media_type = "image/jpeg"

In [ ]:
# message = agent.messages.create(
message = agent.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": image_media_type,
                        "data": base64_frame,
                    },
                },
                {"type": "text", "text": prompt},
            ],
        }
    ],
    output_format=StructuredResponse,
)
print(message)

In [ ]:
print(message.usage.input_tokens)
print(message.usage.output_tokens)
print(message.parsed_output.action[0])

In [40]:
import os
import json
import pandas as pd


input_jsonl = "output/inference_results_gpt-5.4.json"
output_csv = "output/inference_results_gpt-5.4.csv"

rows = []

with open(input_jsonl, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        item = json.loads(line)

        frame_path = item["frame_path"]
        action = item["action"]

        filename = os.path.basename(frame_path)
        filename_wo_ext = os.path.splitext(filename)[0]

        parts = filename_wo_ext.split("_")

        second = int(parts[-2])
        frame_idx = int(parts[-1])
        base_filename = "_".join(parts[:-2])

        rows.append({
            "base_filename": base_filename,
            "second": second,
            # "frame_idx": frame_idx,
            "action": action
        })

df = pd.DataFrame(rows)

df = df.sort_values(
    by=["base_filename", "second"],
    ascending=[True, True]
).reset_index(drop=True)

df.to_csv(output_csv, index=False)